In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import norm

In [2]:
PROJECT_ROOT = Path("..")

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
TABLE_DIR = PROJECT_ROOT / "outputs" / "tables"
FIGURE_DIR = PROJECT_ROOT / "outputs" / "figures"

RETURN_FILE = PROCESSED_DIR / "simple_returns.csv"

CANDIDATE_FILE = (
    TABLE_DIR
    / "portfolio_candidates_600_development.csv"
)

print("Return file exists:", RETURN_FILE.exists())
print("Candidate file exists:", CANDIDATE_FILE.exists())

Return file exists: True
Candidate file exists: True


In [3]:
simple_returns = pd.read_csv(
    RETURN_FILE,
    index_col=0,
    parse_dates=True
).sort_index()

TRAIN_END = "2021-12-31"

TICKERS = [
    "SPY",
    "EFA",
    "EEM",
    "AGG",
    "GLD",
    "XLE",
    "XLK"
]

returns_train = (
    simple_returns
    .loc[:TRAIN_END, TICKERS]
    .copy()
)

print(
    "Development observations:",
    len(returns_train)
)

print(
    "Start:",
    returns_train.index.min()
)

print(
    "End:",
    returns_train.index.max()
)

Development observations: 4309
Start: 2004-11-19 00:00:00
End: 2021-12-31 00:00:00


In [4]:
candidate_df = pd.read_csv(
    CANDIDATE_FILE
)

print(
    "Candidate portfolios:",
    len(candidate_df)
)

candidate_df.head()

Candidate portfolios: 600


,N,Subset,Method,Weights,Expected Return,Volatility,Sharpe Ratio,VaR 95%,ES 95%,VaR 99%,ES 99%,Active Assets,Correct Asset Count
0,2,"SPY, EFA",Equal Weight,0.5000000000;0.5000000000,0.098670,0.200010,0.434453,0.018389,0.030999,0.038302,0.056219,2,True
1,2,"SPY, EFA",GMVP,0.9500000000;0.0500000000,0.117274,0.192672,0.547558,0.017757,0.029940,0.036317,0.053483,2,True
2,2,"SPY, EFA",Maximum Sharpe,0.9500000000;0.0500000000,0.117274,0.192672,0.547558,0.017757,0.029940,0.036317,0.053483,2,True
3,2,"SPY, EFA",Risk Parity,0.5322426735;0.4677573265,0.100003,0.199154,0.443013,0.018272,0.030873,0.038458,0.055980,2,True
4,2,"SPY, EFA",Minimum ES,0.9500000000;0.0500000000,0.117274,0.192672,0.547558,0.017757,0.029940,0.036317,0.053483,2,True


In [5]:
def parse_weights(
    weight_string
):
    return np.array(
        [
            float(x)
            for x in weight_string.split(";")
        ]
    )


candidate_df[
    "Weights Array"
] = candidate_df[
    "Weights"
].apply(
    parse_weights
)

In [6]:
assert len(candidate_df) == 600

assert returns_train.shape == (
    4309,
    7
)

assert (
    returns_train.index.max()
    == pd.Timestamp(
        "2021-12-31"
    )
)

assert (
    candidate_df[
        "Weights Array"
    ]
    .apply(
        lambda w:
        np.isclose(
            w.sum(),
            1.0,
            atol=1e-6
        )
    )
    .all()
)

assert (
    candidate_df[
        "Weights Array"
    ]
    .apply(
        lambda w:
        np.isfinite(w).all()
    )
    .all()
)

print(
    "Notebook 04 -> Notebook 05 "
    "handoff validated."
)

Notebook 04 -> Notebook 05 handoff validated.


In [7]:
candidate_df[
    [
        "Subset",
        "Method",
        "Weights",
        "Weights Array"
    ]
].head()

,Subset,Method,Weights,Weights Array
0,"SPY, EFA",Equal Weight,0.5000000000;0.5000000000,"[0.5, 0.5]"
1,"SPY, EFA",GMVP,0.9500000000;0.0500000000,"[0.95, 0.05]"
2,"SPY, EFA",Maximum Sharpe,0.9500000000;0.0500000000,"[0.95, 0.05]"
3,"SPY, EFA",Risk Parity,0.5322426735;0.4677573265,"[0.5322426735, 0.4677573265]"
4,"SPY, EFA",Minimum ES,0.9500000000;0.0500000000,"[0.95, 0.05]"


In [8]:
def portfolio_returns_from_candidate(
    row,
    return_data
):

    assets = [
        x.strip()
        for x in row[
            "Subset"
        ].split(",")
    ]

    weights = row[
        "Weights Array"
    ]

    subset_returns = (
        return_data[
            assets
        ]
    )

    portfolio_returns = (
        subset_returns.values
        @ weights
    )

    return pd.Series(
        portfolio_returns,
        index=subset_returns.index
    )

In [9]:
best_gmvp = (
    candidate_df[
        candidate_df[
            "Method"
        ] == "GMVP"
    ]
    .sort_values(
        "Volatility"
    )
    .head(1)
)

best_max_sharpe = (
    candidate_df[
        candidate_df[
            "Method"
        ] == "Maximum Sharpe"
    ]
    .sort_values(
        "Sharpe Ratio",
        ascending=False
    )
    .head(1)
)

best_min_es = (
    candidate_df[
        candidate_df[
            "Method"
        ] == "Minimum ES"
    ]
    .sort_values(
        "ES 95%"
    )
    .head(1)
)

best_risk_parity = (
    candidate_df[
        candidate_df[
            "Method"
        ] == "Risk Parity"
    ]
    .sort_values(
        "Sharpe Ratio",
        ascending=False
    )
    .head(1)
)

equal_weight_benchmark = (
    candidate_df[
        (candidate_df["Method"] == "Equal Weight")
        &
        (candidate_df["N"] == 7)
    ]
    .head(1)
)


### Risk Parity Shortlist Rule

Risk Parity portfolios are first generated using the equal-risk-contribution
objective in Notebook 04.

Because multiple asset subsets satisfy the Risk Parity construction principle,
Sharpe ratio is used here only as a secondary ranking criterion to select one
Risk Parity candidate for later risk-model validation.

Sharpe ratio therefore does not define the Risk Parity optimization itself.

In [10]:
shortlist = pd.concat(
    [
        equal_weight_benchmark,
        best_gmvp,
        best_max_sharpe,
        best_risk_parity,
        best_min_es
    ],
    ignore_index=True
)

In [11]:
shortlist[
    [
        "N",
        "Subset",
        "Method",
        "Expected Return",
        "Volatility",
        "Sharpe Ratio",
        "VaR 95%",
        "ES 95%"
    ]
]

,N,Subset,Method,Expected Return,Volatility,Sharpe Ratio,VaR 95%,ES 95%
0,7,"SPY, EFA, EEM, AGG, GLD, XLE, XLK",Equal Weight,0.100306,0.164436,0.538392,0.014697,0.024765
1,2,"SPY, AGG",GMVP,0.045295,0.048037,0.697785,0.003446,0.006114
2,3,"AGG, GLD, XLK",Maximum Sharpe,0.072782,0.062788,0.971625,0.005363,0.008575
3,3,"AGG, GLD, XLK",Risk Parity,0.069322,0.060710,0.947902,0.005190,0.008286
4,2,"SPY, AGG",Minimum ES,0.044859,0.048050,0.688525,0.003450,0.006110


In [12]:
shortlist[
    "Portfolio Label"
] = [
    "Equal Weight Benchmark",
    "Best GMVP",
    "Best Maximum Sharpe",
    "Best Risk Parity",
    "Best Minimum ES"
]

shortlist[
    [
        "Portfolio Label",
        "Subset",
        "Method"
    ]
]

,Portfolio Label,Subset,Method
0,Equal Weight Benchmark,"SPY, EFA, EEM, AGG, GLD, XLE, XLK",Equal Weight
1,Best GMVP,"SPY, AGG",GMVP
2,Best Maximum Sharpe,"AGG, GLD, XLK",Maximum Sharpe
3,Best Risk Parity,"AGG, GLD, XLK",Risk Parity
4,Best Minimum ES,"SPY, AGG",Minimum ES


In [13]:
def historical_var(
    portfolio_returns,
    confidence=0.95
):
    cutoff = np.quantile(
        portfolio_returns,
        1 - confidence
    )

    return -cutoff

In [14]:
def historical_es(
    portfolio_returns,
    confidence=0.95
):
    cutoff = np.quantile(
        portfolio_returns,
        1 - confidence
    )

    tail_returns = portfolio_returns[
        portfolio_returns <= cutoff
    ]

    return -tail_returns.mean()

In [15]:
CONFIDENCE_LEVELS = [
    0.90,
    0.95,
    0.99
]

CONFIDENCE_LEVELS

[0.9, 0.95, 0.99]

In [16]:
historical_risk_rows = []

for _, row in shortlist.iterrows():

    port_returns = (
        portfolio_returns_from_candidate(
            row,
            returns_train
        )
    )

    for confidence in CONFIDENCE_LEVELS:

        var_value = historical_var(
            port_returns.values,
            confidence
        )

        es_value = historical_es(
            port_returns.values,
            confidence
        )

        historical_risk_rows.append({
            "Portfolio":
                row["Portfolio Label"],

            "Subset":
                row["Subset"],

            "Confidence":
                confidence,

            "Historical VaR":
                var_value,

            "Historical ES":
                es_value
        })

historical_risk = pd.DataFrame(
    historical_risk_rows
)

historical_risk

,Portfolio,Subset,Confidence,Historical VaR,Historical ES
0,Equal Weight Benchmark,"SPY, EFA, EEM, AGG, GLD, XLE, XLK",0.90,0.009836,0.018452
1,Equal Weight Benchmark,"SPY, EFA, EEM, AGG, GLD, XLE, XLK",0.95,0.014697,0.024765
2,Equal Weight Benchmark,"SPY, EFA, EEM, AGG, GLD, XLE, XLK",0.99,0.030285,0.044691
3,Best GMVP,"SPY, AGG",0.90,0.002343,0.004494
4,Best GMVP,"SPY, AGG",0.95,0.003446,0.006114
5,Best GMVP,"SPY, AGG",0.99,0.006490,0.012690
6,Best Maximum Sharpe,"AGG, GLD, XLK",0.90,0.003628,0.006506
7,Best Maximum Sharpe,"AGG, GLD, XLK",0.95,0.005363,0.008575
8,Best Maximum Sharpe,"AGG, GLD, XLK",0.99,0.009713,0.015404
9,Best Risk Parity,"AGG, GLD, XLK",0.90,0.003577,0.006291


In [17]:
historical_risk[
    "ES >= VaR"
] = (
    historical_risk[
        "Historical ES"
    ]
    >=
    historical_risk[
        "Historical VaR"
    ]
)

historical_risk[
    "ES >= VaR"
].value_counts()

ES >= VaR
True    15
Name: count, dtype: int64

In [18]:
historical_var_table = (
    historical_risk
    .pivot(
        index="Portfolio",
        columns="Confidence",
        values="Historical VaR"
    )
)

historical_var_table.columns = [
    "VaR 90%",
    "VaR 95%",
    "VaR 99%"
]

historical_var_table

,VaR 90%,VaR 95%,VaR 99%
Portfolio,,,
Best GMVP,0.002343,0.003446,0.006490
Best Maximum Sharpe,0.003628,0.005363,0.009713
Best Minimum ES,0.002355,0.003450,0.006474
Best Risk Parity,0.003577,0.005190,0.009393
Equal Weight Benchmark,0.009836,0.014697,0.030285


In [19]:
historical_es_table = (
    historical_risk
    .pivot(
        index="Portfolio",
        columns="Confidence",
        values="Historical ES"
    )
)

historical_es_table.columns = [
    "ES 90%",
    "ES 95%",
    "ES 99%"
]

historical_es_table

,ES 90%,ES 95%,ES 99%
Portfolio,,,
Best GMVP,0.004494,0.006114,0.012690
Best Maximum Sharpe,0.006506,0.008575,0.015404
Best Minimum ES,0.004497,0.006110,0.012667
Best Risk Parity,0.006291,0.008286,0.014583
Equal Weight Benchmark,0.018452,0.024765,0.044691


In [20]:
historical_summary = (
    historical_var_table
    .join(
        historical_es_table
    )
)

historical_summary

,VaR 90%,VaR 95%,VaR 99%,ES 90%,ES 95%,ES 99%
Portfolio,,,,,,
Best GMVP,0.002343,0.003446,0.006490,0.004494,0.006114,0.012690
Best Maximum Sharpe,0.003628,0.005363,0.009713,0.006506,0.008575,0.015404
Best Minimum ES,0.002355,0.003450,0.006474,0.004497,0.006110,0.012667
Best Risk Parity,0.003577,0.005190,0.009393,0.006291,0.008286,0.014583
Equal Weight Benchmark,0.009836,0.014697,0.030285,0.018452,0.024765,0.044691


In [21]:
historical_summary_percent = (
    historical_summary
    * 100
)

historical_summary_percent.round(3)

,VaR 90%,VaR 95%,VaR 99%,ES 90%,ES 95%,ES 99%
Portfolio,,,,,,
Best GMVP,0.234,0.345,0.649,0.449,0.611,1.269
Best Maximum Sharpe,0.363,0.536,0.971,0.651,0.858,1.540
Best Minimum ES,0.236,0.345,0.647,0.450,0.611,1.267
Best Risk Parity,0.358,0.519,0.939,0.629,0.829,1.458
Equal Weight Benchmark,0.984,1.470,3.029,1.845,2.477,4.469


In [22]:
def parametric_var(
    portfolio_returns,
    confidence=0.95
):

    mean_return = np.mean(
        portfolio_returns
    )

    volatility = np.std(
        portfolio_returns,
        ddof=1
    )

    z = norm.ppf(
        1 - confidence
    )

    var_value = -(
        mean_return
        +
        z * volatility
    )

    return var_value

In [23]:
def parametric_es(
    portfolio_returns,
    confidence=0.95
):

    mean_return = np.mean(
        portfolio_returns
    )

    volatility = np.std(
        portfolio_returns,
        ddof=1
    )

    z = norm.ppf(
        confidence
    )

    es_value = (
        -mean_return
        +
        volatility
        * norm.pdf(z)
        / (1 - confidence)
    )

    return es_value

In [24]:
parametric_risk_rows = []

for _, row in shortlist.iterrows():

    port_returns = (
        portfolio_returns_from_candidate(
            row,
            returns_train
        )
    )

    for confidence in CONFIDENCE_LEVELS:

        var_value = parametric_var(
            port_returns.values,
            confidence
        )

        es_value = parametric_es(
            port_returns.values,
            confidence
        )

        parametric_risk_rows.append({
            "Portfolio":
                row["Portfolio Label"],

            "Subset":
                row["Subset"],

            "Confidence":
                confidence,

            "Parametric VaR":
                var_value,

            "Parametric ES":
                es_value
        })

parametric_risk = pd.DataFrame(
    parametric_risk_rows
)

parametric_risk

,Portfolio,Subset,Confidence,Parametric VaR,Parametric ES
0,Equal Weight Benchmark,"SPY, EFA, EEM, AGG, GLD, XLE, XLK",0.90,0.012910,0.017827
1,Equal Weight Benchmark,"SPY, EFA, EEM, AGG, GLD, XLE, XLK",0.95,0.016683,0.021023
2,Equal Weight Benchmark,"SPY, EFA, EEM, AGG, GLD, XLE, XLK",0.99,0.023760,0.027279
3,Best GMVP,"SPY, AGG",0.90,0.003563,0.004945
4,Best GMVP,"SPY, AGG",0.95,0.004624,0.005844
5,Best GMVP,"SPY, AGG",0.99,0.006614,0.007603
6,Best Maximum Sharpe,"AGG, GLD, XLK",0.90,0.004707,0.006553
7,Best Maximum Sharpe,"AGG, GLD, XLK",0.95,0.006124,0.007753
8,Best Maximum Sharpe,"AGG, GLD, XLK",0.99,0.008781,0.010102
9,Best Risk Parity,"AGG, GLD, XLK",0.90,0.004549,0.006331


In [25]:
parametric_risk[
    "ES >= VaR"
] = (
    parametric_risk[
        "Parametric ES"
    ]
    >=
    parametric_risk[
        "Parametric VaR"
    ]
)

parametric_risk[
    "ES >= VaR"
].value_counts()

ES >= VaR
True    15
Name: count, dtype: int64

In [26]:
parametric_var_table = (
    parametric_risk
    .pivot(
        index="Portfolio",
        columns="Confidence",
        values="Parametric VaR"
    )
)

parametric_var_table.columns = [
    "Parametric VaR 90%",
    "Parametric VaR 95%",
    "Parametric VaR 99%"
]

parametric_es_table = (
    parametric_risk
    .pivot(
        index="Portfolio",
        columns="Confidence",
        values="Parametric ES"
    )
)

parametric_es_table.columns = [
    "Parametric ES 90%",
    "Parametric ES 95%",
    "Parametric ES 99%"
]

parametric_summary = (
    parametric_var_table
    .join(
        parametric_es_table
    )
)

parametric_summary

,Parametric VaR 90%,Parametric VaR 95%,Parametric VaR 99%,Parametric ES 90%,Parametric ES 95%,Parametric ES 99%
Portfolio,,,,,,
Best GMVP,0.003563,0.004624,0.006614,0.004945,0.005844,0.007603
Best Maximum Sharpe,0.004707,0.006124,0.008781,0.006553,0.007753,0.010102
Best Minimum ES,0.003564,0.004624,0.006614,0.004946,0.005844,0.007604
Best Risk Parity,0.004549,0.005917,0.008482,0.006331,0.007490,0.009758
Equal Weight Benchmark,0.012910,0.016683,0.023760,0.017827,0.021023,0.027279


In [27]:
historical_vs_parametric = pd.concat(
    [
        historical_var_table,
        parametric_var_table
    ],
    axis=1
)

historical_vs_parametric

,VaR 90%,VaR 95%,VaR 99%,Parametric VaR 90%,Parametric VaR 95%,Parametric VaR 99%
Portfolio,,,,,,
Best GMVP,0.002343,0.003446,0.006490,0.003563,0.004624,0.006614
Best Maximum Sharpe,0.003628,0.005363,0.009713,0.004707,0.006124,0.008781
Best Minimum ES,0.002355,0.003450,0.006474,0.003564,0.004624,0.006614
Best Risk Parity,0.003577,0.005190,0.009393,0.004549,0.005917,0.008482
Equal Weight Benchmark,0.009836,0.014697,0.030285,0.012910,0.016683,0.023760


In [28]:
(
    historical_vs_parametric
    * 100
).round(3)

,VaR 90%,VaR 95%,VaR 99%,Parametric VaR 90%,Parametric VaR 95%,Parametric VaR 99%
Portfolio,,,,,,
Best GMVP,0.234,0.345,0.649,0.356,0.462,0.661
Best Maximum Sharpe,0.363,0.536,0.971,0.471,0.612,0.878
Best Minimum ES,0.236,0.345,0.647,0.356,0.462,0.661
Best Risk Parity,0.358,0.519,0.939,0.455,0.592,0.848
Equal Weight Benchmark,0.984,1.470,3.029,1.291,1.668,2.376


In [29]:
MC_SIMULATIONS = 100_000

RANDOM_SEED = 42

np.random.seed(
    RANDOM_SEED
)

print(
    "Monte Carlo simulations:",
    MC_SIMULATIONS
)

Monte Carlo simulations: 100000


In [30]:
def monte_carlo_portfolio_returns(
    row,
    return_data,
    n_simulations=100_000,
    random_seed=42
):

    assets = [
        x.strip()
        for x in row[
            "Subset"
        ].split(",")
    ]

    weights = row[
        "Weights Array"
    ]

    subset_returns = (
        return_data[
            assets
        ]
    )

    mean_vector = (
        subset_returns.mean().values
    )

    covariance_matrix = (
        subset_returns.cov().values
    )

    rng = np.random.default_rng(
        random_seed
    )

    simulated_asset_returns = (
        rng.multivariate_normal(
            mean=mean_vector,
            cov=covariance_matrix,
            size=n_simulations
        )
    )

    simulated_portfolio_returns = (
        simulated_asset_returns
        @ weights
    )

    return simulated_portfolio_returns

In [31]:
def monte_carlo_var(
    simulated_returns,
    confidence=0.95
):

    cutoff = np.quantile(
        simulated_returns,
        1 - confidence
    )

    return -cutoff


def monte_carlo_es(
    simulated_returns,
    confidence=0.95
):

    cutoff = np.quantile(
        simulated_returns,
        1 - confidence
    )

    tail_returns = simulated_returns[
        simulated_returns <= cutoff
    ]

    return -tail_returns.mean()

In [32]:
monte_carlo_risk_rows = []

for i, row in shortlist.iterrows():

    print(
        "Simulating:",
        row[
            "Portfolio Label"
        ]
    )

    simulated_returns = (
        monte_carlo_portfolio_returns(
            row,
            returns_train,
            n_simulations=MC_SIMULATIONS,
            random_seed=RANDOM_SEED + i
        )
    )

    for confidence in CONFIDENCE_LEVELS:

        var_value = monte_carlo_var(
            simulated_returns,
            confidence
        )

        es_value = monte_carlo_es(
            simulated_returns,
            confidence
        )

        monte_carlo_risk_rows.append({
            "Portfolio":
                row[
                    "Portfolio Label"
                ],

            "Subset":
                row[
                    "Subset"
                ],

            "Confidence":
                confidence,

            "Monte Carlo VaR":
                var_value,

            "Monte Carlo ES":
                es_value
        })


monte_carlo_risk = pd.DataFrame(
    monte_carlo_risk_rows
)

monte_carlo_risk

Simulating: Equal Weight Benchmark
Simulating: Best GMVP
Simulating: Best Maximum Sharpe
Simulating: Best Risk Parity
Simulating: Best Minimum ES


,Portfolio,Subset,Confidence,Monte Carlo VaR,Monte Carlo ES
0,Equal Weight Benchmark,"SPY, EFA, EEM, AGG, GLD, XLE, XLK",0.90,0.012836,0.017803
1,Equal Weight Benchmark,"SPY, EFA, EEM, AGG, GLD, XLE, XLK",0.95,0.016666,0.021016
2,Equal Weight Benchmark,"SPY, EFA, EEM, AGG, GLD, XLE, XLK",0.99,0.023723,0.027230
3,Best GMVP,"SPY, AGG",0.90,0.003553,0.004928
4,Best GMVP,"SPY, AGG",0.95,0.004626,0.005813
5,Best GMVP,"SPY, AGG",0.99,0.006536,0.007523
6,Best Maximum Sharpe,"AGG, GLD, XLK",0.90,0.004685,0.006556
7,Best Maximum Sharpe,"AGG, GLD, XLK",0.95,0.006130,0.007767
8,Best Maximum Sharpe,"AGG, GLD, XLK",0.99,0.008769,0.010148
9,Best Risk Parity,"AGG, GLD, XLK",0.90,0.004538,0.006334


In [33]:
monte_carlo_risk[
    "ES >= VaR"
] = (
    monte_carlo_risk[
        "Monte Carlo ES"
    ]
    >=
    monte_carlo_risk[
        "Monte Carlo VaR"
    ]
)

monte_carlo_risk[
    "ES >= VaR"
].value_counts()

ES >= VaR
True    15
Name: count, dtype: int64

In [34]:
monte_carlo_var_table = (
    monte_carlo_risk
    .pivot(
        index="Portfolio",
        columns="Confidence",
        values="Monte Carlo VaR"
    )
)

monte_carlo_var_table.columns = [
    "Monte Carlo VaR 90%",
    "Monte Carlo VaR 95%",
    "Monte Carlo VaR 99%"
]

monte_carlo_var_table

,Monte Carlo VaR 90%,Monte Carlo VaR 95%,Monte Carlo VaR 99%
Portfolio,,,
Best GMVP,0.003553,0.004626,0.006536
Best Maximum Sharpe,0.004685,0.006130,0.008769
Best Minimum ES,0.003551,0.004623,0.006624
Best Risk Parity,0.004538,0.005905,0.008485
Equal Weight Benchmark,0.012836,0.016666,0.023723


In [35]:
monte_carlo_es_table = (
    monte_carlo_risk
    .pivot(
        index="Portfolio",
        columns="Confidence",
        values="Monte Carlo ES"
    )
)

monte_carlo_es_table.columns = [
    "Monte Carlo ES 90%",
    "Monte Carlo ES 95%",
    "Monte Carlo ES 99%"
]

monte_carlo_es_table

,Monte Carlo ES 90%,Monte Carlo ES 95%,Monte Carlo ES 99%
Portfolio,,,
Best GMVP,0.004928,0.005813,0.007523
Best Maximum Sharpe,0.006556,0.007767,0.010148
Best Minimum ES,0.004948,0.005855,0.007598
Best Risk Parity,0.006334,0.007505,0.009756
Equal Weight Benchmark,0.017803,0.021016,0.027230


In [36]:
monte_carlo_summary = (
    monte_carlo_var_table
    .join(
        monte_carlo_es_table
    )
)

monte_carlo_summary

,Monte Carlo VaR 90%,Monte Carlo VaR 95%,Monte Carlo VaR 99%,Monte Carlo ES 90%,Monte Carlo ES 95%,Monte Carlo ES 99%
Portfolio,,,,,,
Best GMVP,0.003553,0.004626,0.006536,0.004928,0.005813,0.007523
Best Maximum Sharpe,0.004685,0.006130,0.008769,0.006556,0.007767,0.010148
Best Minimum ES,0.003551,0.004623,0.006624,0.004948,0.005855,0.007598
Best Risk Parity,0.004538,0.005905,0.008485,0.006334,0.007505,0.009756
Equal Weight Benchmark,0.012836,0.016666,0.023723,0.017803,0.021016,0.027230


In [37]:
var_model_comparison = pd.concat(
    [
        historical_var_table,
        parametric_var_table,
        monte_carlo_var_table
    ],
    axis=1
)

(
    var_model_comparison
    * 100
).round(3)

,VaR 90%,VaR 95%,VaR 99%,Parametric VaR 90%,Parametric VaR 95%,Parametric VaR 99%,Monte Carlo VaR 90%,Monte Carlo VaR 95%,Monte Carlo VaR 99%
Portfolio,,,,,,,,,
Best GMVP,0.234,0.345,0.649,0.356,0.462,0.661,0.355,0.463,0.654
Best Maximum Sharpe,0.363,0.536,0.971,0.471,0.612,0.878,0.468,0.613,0.877
Best Minimum ES,0.236,0.345,0.647,0.356,0.462,0.661,0.355,0.462,0.662
Best Risk Parity,0.358,0.519,0.939,0.455,0.592,0.848,0.454,0.590,0.849
Equal Weight Benchmark,0.984,1.470,3.029,1.291,1.668,2.376,1.284,1.667,2.372


In [38]:
var_99_plot = pd.DataFrame({
    "Historical": historical_var_table["VaR 99%"],
    "Parametric Normal": parametric_var_table["Parametric VaR 99%"],
    "Monte Carlo Normal": monte_carlo_var_table["Monte Carlo VaR 99%"]
})

(
    var_99_plot
    * 100
).plot(
    kind="bar",
    figsize=(10, 6)
)

plt.ylabel("One-Day VaR (%)")
plt.xlabel("Portfolio")
plt.title("99% VaR Comparison Across Risk Models")
plt.xticks(rotation=30, ha="right")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()

plt.savefig(
    FIGURE_DIR / "var_99_model_comparison.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

C:\Users\Jani\AppData\Local\Temp\ipykernel_15712\24685338.py:28: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [39]:
es_99_plot = pd.DataFrame({
    "Historical": historical_es_table["ES 99%"],
    "Parametric Normal": parametric_es_table["Parametric ES 99%"],
    "Monte Carlo Normal": monte_carlo_es_table["Monte Carlo ES 99%"]
})

(
    es_99_plot
    * 100
).plot(
    kind="bar",
    figsize=(10, 6)
)

plt.ylabel("One-Day Expected Shortfall (%)")
plt.xlabel("Portfolio")
plt.title("99% Expected Shortfall Comparison Across Risk Models")
plt.xticks(rotation=30, ha="right")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()

plt.savefig(
    FIGURE_DIR / "es_99_model_comparison.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

C:\Users\Jani\AppData\Local\Temp\ipykernel_15712\2737287902.py:28: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [40]:
var_model_gap = pd.DataFrame({
    "Historical VaR 99%": historical_var_table["VaR 99%"],
    "Parametric VaR 99%": parametric_var_table["Parametric VaR 99%"],
    "Monte Carlo VaR 99%": monte_carlo_var_table["Monte Carlo VaR 99%"]
})

var_model_gap[
    "Historical - Parametric"
] = (
    var_model_gap["Historical VaR 99%"]
    -
    var_model_gap["Parametric VaR 99%"]
)

var_model_gap[
    "Historical - Monte Carlo"
] = (
    var_model_gap["Historical VaR 99%"]
    -
    var_model_gap["Monte Carlo VaR 99%"]
)

(
    var_model_gap
    * 100
).round(3)

,Historical VaR 99%,Parametric VaR 99%,Monte Carlo VaR 99%,Historical - Parametric,Historical - Monte Carlo
Portfolio,,,,,
Best GMVP,0.649,0.661,0.654,-0.012,-0.005
Best Maximum Sharpe,0.971,0.878,0.877,0.093,0.094
Best Minimum ES,0.647,0.661,0.662,-0.014,-0.015
Best Risk Parity,0.939,0.848,0.849,0.091,0.091
Equal Weight Benchmark,3.029,2.376,2.372,0.652,0.656


In [41]:
var_model_comparison_95 = pd.DataFrame({

    "Historical VaR 95%":
        historical_var_table[
            "VaR 95%"
        ],

    "Parametric VaR 95%":
        parametric_var_table[
            "Parametric VaR 95%"
        ],

    "Monte Carlo VaR 95%":
        monte_carlo_var_table[
            "Monte Carlo VaR 95%"
        ]
})

(
    var_model_comparison_95
    * 100
).round(3)

,Historical VaR 95%,Parametric VaR 95%,Monte Carlo VaR 95%
Portfolio,,,
Best GMVP,0.345,0.462,0.463
Best Maximum Sharpe,0.536,0.612,0.613
Best Minimum ES,0.345,0.462,0.462
Best Risk Parity,0.519,0.592,0.590
Equal Weight Benchmark,1.470,1.668,1.667


In [42]:
es_model_comparison_95 = pd.DataFrame({

    "Historical ES 95%":
        historical_es_table[
            "ES 95%"
        ],

    "Parametric ES 95%":
        parametric_es_table[
            "Parametric ES 95%"
        ],

    "Monte Carlo ES 95%":
        monte_carlo_es_table[
            "Monte Carlo ES 95%"
        ]
})

(
    es_model_comparison_95
    * 100
).round(3)

,Historical ES 95%,Parametric ES 95%,Monte Carlo ES 95%
Portfolio,,,
Best GMVP,0.611,0.584,0.581
Best Maximum Sharpe,0.858,0.775,0.777
Best Minimum ES,0.611,0.584,0.586
Best Risk Parity,0.829,0.749,0.751
Equal Weight Benchmark,2.477,2.102,2.102


## Monte Carlo Assumption

The Monte Carlo simulation assumes multivariate normally distributed daily asset
returns using the historical training-period mean vector and covariance matrix.

This preserves linear dependence and covariance structure but does not fully
capture skewness, excess kurtosis or nonlinear tail dependence.

For this reason, Monte Carlo VaR is compared with Historical VaR and EVT-based
tail estimates rather than treated as a complete description of extreme risk.

In [43]:
from scipy.stats import genpareto

In [44]:
EVT_THRESHOLD_QUANTILE = 0.95

print(
    "EVT threshold quantile:",
    EVT_THRESHOLD_QUANTILE
)

EVT threshold quantile: 0.95


In [45]:
def fit_evt_gpd(
    portfolio_returns,
    threshold_quantile=0.95
):

    losses = -np.asarray(
        portfolio_returns
    )

    threshold = np.quantile(
        losses,
        threshold_quantile
    )

    exceedances = losses[
        losses > threshold
    ] - threshold

    shape, loc, scale = genpareto.fit(
        exceedances,
        floc=0
    )

    return {
        "losses": losses,
        "threshold": threshold,
        "exceedances": exceedances,
        "shape": shape,
        "scale": scale
    }

In [46]:
benchmark_row = shortlist[
    shortlist[
        "Portfolio Label"
    ] == "Equal Weight Benchmark"
].iloc[0]

benchmark_returns = (
    portfolio_returns_from_candidate(
        benchmark_row,
        returns_train
    )
)

benchmark_evt = fit_evt_gpd(
    benchmark_returns.values,
    threshold_quantile=EVT_THRESHOLD_QUANTILE
)

print(
    "Threshold:",
    f"{benchmark_evt['threshold']:.4%}"
)

print(
    "Number of exceedances:",
    len(
        benchmark_evt[
            "exceedances"
        ]
    )
)

print(
    "Shape parameter:",
    round(
        benchmark_evt[
            "shape"
        ],
        4
    )
)

print(
    "Scale parameter:",
    round(
        benchmark_evt[
            "scale"
        ],
        6
    )
)

Threshold: 1.4697%
Number of exceedances: 216
Shape parameter: 0.276
Scale parameter: 0.0074


In [47]:
def evt_var(
    evt_fit,
    confidence
):

    losses = evt_fit["losses"]
    threshold = evt_fit["threshold"]
    exceedances = evt_fit["exceedances"]
    shape = evt_fit["shape"]
    scale = evt_fit["scale"]

    n_total = len(losses)
    n_exceed = len(exceedances)

    tail_ratio = (
        n_exceed
        /
        (
            n_total
            * (1 - confidence)
        )
    )

    if abs(shape) < 1e-8:

        var_value = (
            threshold
            +
            scale
            * np.log(tail_ratio)
        )

    else:

        var_value = (
            threshold
            +
            scale
            / shape
            * (
                tail_ratio ** shape
                - 1
            )
        )

    return var_value

In [48]:
def evt_es(
    evt_fit,
    confidence
):

    shape = evt_fit["shape"]
    scale = evt_fit["scale"]
    threshold = evt_fit["threshold"]

    if shape >= 1:
        return np.nan

    var_value = evt_var(
        evt_fit,
        confidence
    )

    es_value = (
        var_value
        +
        scale
        -
        shape * threshold
    ) / (
        1 - shape
    )

    return es_value

In [49]:
EVT_CONFIDENCE_LEVELS = [
    0.99,
    0.995
]

for confidence in EVT_CONFIDENCE_LEVELS:

    var_value = evt_var(
        benchmark_evt,
        confidence
    )

    es_value = evt_es(
        benchmark_evt,
        confidence
    )

    print(
        f"EVT VaR {confidence:.1%}:",
        f"{var_value:.4%}"
    )

    print(
        f"EVT ES {confidence:.1%}:",
        f"{es_value:.4%}"
    )

    print()

EVT VaR 99.0%: 2.9722%
EVT ES 99.0%: 4.5670%

EVT VaR 99.5%: 3.8542%
EVT ES 99.5%: 5.7853%



In [50]:
evt_rows = []

for _, row in shortlist.iterrows():

    port_returns = (
        portfolio_returns_from_candidate(
            row,
            returns_train
        )
    )

    evt_fit = fit_evt_gpd(
        port_returns.values,
        threshold_quantile=EVT_THRESHOLD_QUANTILE
    )

    for confidence in EVT_CONFIDENCE_LEVELS:

        var_value = evt_var(
            evt_fit,
            confidence
        )

        es_value = evt_es(
            evt_fit,
            confidence
        )

        evt_rows.append({
            "Portfolio":
                row["Portfolio Label"],

            "Subset":
                row["Subset"],

            "Confidence":
                confidence,

            "Threshold":
                evt_fit["threshold"],

            "Exceedances":
                len(
                    evt_fit[
                        "exceedances"
                    ]
                ),

            "Shape":
                evt_fit["shape"],

            "Scale":
                evt_fit["scale"],

            "EVT VaR":
                var_value,

            "EVT ES":
                es_value
        })


evt_risk = pd.DataFrame(
    evt_rows
)

evt_risk

,Portfolio,Subset,Confidence,Threshold,Exceedances,Shape,Scale,EVT VaR,EVT ES
0,Equal Weight Benchmark,"SPY, EFA, EEM, AGG, GLD, XLE, XLK",0.990,0.014697,216,0.275992,0.007400,0.029722,0.045670
1,Equal Weight Benchmark,"SPY, EFA, EEM, AGG, GLD, XLE, XLK",0.995,0.014697,216,0.275992,0.007400,0.038542,0.057853
2,Best GMVP,"SPY, AGG",0.990,0.003446,216,0.514287,0.001290,0.006684,0.012769
3,Best GMVP,"SPY, AGG",0.995,0.003446,216,0.514287,0.001290,0.009145,0.017836
4,Best Maximum Sharpe,"AGG, GLD, XLK",0.990,0.005363,216,0.350021,0.002061,0.009826,0.015399
5,Best Maximum Sharpe,"AGG, GLD, XLK",0.995,0.005363,216,0.350021,0.002061,0.012667,0.019771
6,Best Risk Parity,"AGG, GLD, XLK",0.990,0.005190,216,0.287451,0.002142,0.009583,0.014362
7,Best Risk Parity,"AGG, GLD, XLK",0.995,0.005190,216,0.287451,0.002142,0.012195,0.018028
8,Best Minimum ES,"SPY, AGG",0.990,0.003450,216,0.517789,0.001280,0.006673,0.012788
9,Best Minimum ES,"SPY, AGG",0.995,0.003450,216,0.517789,0.001280,0.009132,0.017886


In [51]:
evt_risk[
    "ES >= VaR"
] = (
    evt_risk[
        "EVT ES"
    ]
    >=
    evt_risk[
        "EVT VaR"
    ]
)

print(
    evt_risk[
        "ES >= VaR"
    ].value_counts()
)

print(
    "\nNumber of EVT rows:",
    len(evt_risk)
)

ES >= VaR
True    10
Name: count, dtype: int64

Number of EVT rows: 10


In [52]:
evt_parameters = (
    evt_risk[
        [
            "Portfolio",
            "Threshold",
            "Exceedances",
            "Shape",
            "Scale"
        ]
    ]
    .drop_duplicates(
        subset="Portfolio"
    )
    .set_index(
        "Portfolio"
    )
)

evt_parameters

,Threshold,Exceedances,Shape,Scale
Portfolio,,,,
Equal Weight Benchmark,0.014697,216,0.275992,0.007400
Best GMVP,0.003446,216,0.514287,0.001290
Best Maximum Sharpe,0.005363,216,0.350021,0.002061
Best Risk Parity,0.005190,216,0.287451,0.002142
Best Minimum ES,0.003450,216,0.517789,0.001280


In [53]:
evt_var_table = (
    evt_risk
    .pivot(
        index="Portfolio",
        columns="Confidence",
        values="EVT VaR"
    )
)

evt_var_table.columns = [
    "EVT VaR 99%",
    "EVT VaR 99.5%"
]

evt_var_table

,EVT VaR 99%,EVT VaR 99.5%
Portfolio,,
Best GMVP,0.006684,0.009145
Best Maximum Sharpe,0.009826,0.012667
Best Minimum ES,0.006673,0.009132
Best Risk Parity,0.009583,0.012195
Equal Weight Benchmark,0.029722,0.038542


In [54]:
evt_es_table = (
    evt_risk
    .pivot(
        index="Portfolio",
        columns="Confidence",
        values="EVT ES"
    )
)

evt_es_table.columns = [
    "EVT ES 99%",
    "EVT ES 99.5%"
]

evt_es_table

,EVT ES 99%,EVT ES 99.5%
Portfolio,,
Best GMVP,0.012769,0.017836
Best Maximum Sharpe,0.015399,0.019771
Best Minimum ES,0.012788,0.017886
Best Risk Parity,0.014362,0.018028
Equal Weight Benchmark,0.045670,0.057853


In [55]:
evt_model_comparison_99 = pd.DataFrame({

    "Historical VaR 99%":
        historical_var_table[
            "VaR 99%"
        ],

    "Parametric VaR 99%":
        parametric_var_table[
            "Parametric VaR 99%"
        ],

    "Monte Carlo VaR 99%":
        monte_carlo_var_table[
            "Monte Carlo VaR 99%"
        ],

    "EVT VaR 99%":
        evt_var_table[
            "EVT VaR 99%"
        ],

    "Historical ES 99%":
        historical_es_table[
            "ES 99%"
        ],

    "Parametric ES 99%":
        parametric_es_table[
            "Parametric ES 99%"
        ],

    "Monte Carlo ES 99%":
        monte_carlo_es_table[
            "Monte Carlo ES 99%"
        ],

    "EVT ES 99%":
        evt_es_table[
            "EVT ES 99%"
        ]
})

(
    evt_model_comparison_99
    * 100
).round(3)

,Historical VaR 99%,Parametric VaR 99%,Monte Carlo VaR 99%,EVT VaR 99%,Historical ES 99%,Parametric ES 99%,Monte Carlo ES 99%,EVT ES 99%
Portfolio,,,,,,,,
Best GMVP,0.649,0.661,0.654,0.668,1.269,0.760,0.752,1.277
Best Maximum Sharpe,0.971,0.878,0.877,0.983,1.540,1.010,1.015,1.540
Best Minimum ES,0.647,0.661,0.662,0.667,1.267,0.760,0.760,1.279
Best Risk Parity,0.939,0.848,0.849,0.958,1.458,0.976,0.976,1.436
Equal Weight Benchmark,3.029,2.376,2.372,2.972,4.469,2.728,2.723,4.567


In [56]:
EVT_THRESHOLD_LEVELS = [
    0.90,
    0.925,
    0.95,
    0.975
]

EVT_THRESHOLD_LEVELS

[0.9, 0.925, 0.95, 0.975]

In [57]:
evt_sensitivity_rows = []

for _, row in shortlist.iterrows():

    port_returns = (
        portfolio_returns_from_candidate(
            row,
            returns_train
        )
    )

    for threshold_quantile in EVT_THRESHOLD_LEVELS:

        evt_fit = fit_evt_gpd(
            port_returns.values,
            threshold_quantile=threshold_quantile
        )

        var_99 = evt_var(
            evt_fit,
            confidence=0.99
        )

        es_99 = evt_es(
            evt_fit,
            confidence=0.99
        )

        evt_sensitivity_rows.append({
            "Portfolio":
                row["Portfolio Label"],

            "Threshold Quantile":
                threshold_quantile,

            "Threshold":
                evt_fit["threshold"],

            "Exceedances":
                len(
                    evt_fit["exceedances"]
                ),

            "Shape":
                evt_fit["shape"],

            "Scale":
                evt_fit["scale"],

            "EVT VaR 99%":
                var_99,

            "EVT ES 99%":
                es_99
        })


evt_sensitivity = pd.DataFrame(
    evt_sensitivity_rows
)

evt_sensitivity

,Portfolio,Threshold Quantile,Threshold,Exceedances,Shape,Scale,EVT VaR 99%,EVT ES 99%
0,Equal Weight Benchmark,0.900,0.009836,431,0.223649,0.006701,0.030021,0.044468
1,Equal Weight Benchmark,0.925,0.012084,324,0.312405,0.006198,0.029506,0.046435
2,Equal Weight Benchmark,0.950,0.014697,216,0.275992,0.007400,0.029722,0.045670
3,Equal Weight Benchmark,0.975,0.020521,108,0.253940,0.009064,0.029902,0.045244
4,Best GMVP,0.900,0.002343,431,0.325045,0.001376,0.007057,0.011366
5,Best GMVP,0.925,0.002856,324,0.419786,0.001258,0.006848,0.011903
6,Best GMVP,0.950,0.003446,216,0.514287,0.001290,0.006684,0.012769
7,Best GMVP,0.975,0.004603,108,0.634799,0.001523,0.006503,0.013976
8,Best Maximum Sharpe,0.900,0.003628,431,0.211863,0.002226,0.010235,0.014835
9,Best Maximum Sharpe,0.925,0.004422,324,0.299660,0.002000,0.009965,0.015193


In [58]:
evt_sensitivity[
    [
        "Portfolio",
        "Threshold Quantile",
        "Exceedances"
    ]
]

,Portfolio,Threshold Quantile,Exceedances
0,Equal Weight Benchmark,0.900,431
1,Equal Weight Benchmark,0.925,324
2,Equal Weight Benchmark,0.950,216
3,Equal Weight Benchmark,0.975,108
4,Best GMVP,0.900,431
5,Best GMVP,0.925,324
6,Best GMVP,0.950,216
7,Best GMVP,0.975,108
8,Best Maximum Sharpe,0.900,431
9,Best Maximum Sharpe,0.925,324


In [59]:
evt_shape_sensitivity = (
    evt_sensitivity
    .pivot(
        index="Portfolio",
        columns="Threshold Quantile",
        values="Shape"
    )
)

evt_shape_sensitivity.columns = [
    "Shape 90%",
    "Shape 92.5%",
    "Shape 95%",
    "Shape 97.5%"
]

evt_shape_sensitivity.round(4)

,Shape 90%,Shape 92.5%,Shape 95%,Shape 97.5%
Portfolio,,,,
Best GMVP,0.3250,0.4198,0.5143,0.6348
Best Maximum Sharpe,0.2119,0.2997,0.3500,0.3714
Best Minimum ES,0.3270,0.4163,0.5178,0.6564
Best Risk Parity,0.2200,0.2280,0.2875,0.3765
Equal Weight Benchmark,0.2236,0.3124,0.2760,0.2539


In [60]:
evt_var_sensitivity = (
    evt_sensitivity
    .pivot(
        index="Portfolio",
        columns="Threshold Quantile",
        values="EVT VaR 99%"
    )
)

evt_var_sensitivity.columns = [
    "VaR 99% | Threshold 90%",
    "VaR 99% | Threshold 92.5%",
    "VaR 99% | Threshold 95%",
    "VaR 99% | Threshold 97.5%"
]

(
    evt_var_sensitivity
    * 100
).round(3)

,VaR 99% | Threshold 90%,VaR 99% | Threshold 92.5%,VaR 99% | Threshold 95%,VaR 99% | Threshold 97.5%
Portfolio,,,,
Best GMVP,0.706,0.685,0.668,0.650
Best Maximum Sharpe,1.024,0.997,0.983,0.978
Best Minimum ES,0.705,0.685,0.667,0.647
Best Risk Parity,0.980,0.977,0.958,0.938
Equal Weight Benchmark,3.002,2.951,2.972,2.990


In [61]:
evt_es_sensitivity = (
    evt_sensitivity
    .pivot(
        index="Portfolio",
        columns="Threshold Quantile",
        values="EVT ES 99%"
    )
)

evt_es_sensitivity.columns = [
    "ES 99% | Threshold 90%",
    "ES 99% | Threshold 92.5%",
    "ES 99% | Threshold 95%",
    "ES 99% | Threshold 97.5%"
]

(
    evt_es_sensitivity
    * 100
).round(3)

,ES 99% | Threshold 90%,ES 99% | Threshold 92.5%,ES 99% | Threshold 95%,ES 99% | Threshold 97.5%
Portfolio,,,,
Best GMVP,1.137,1.190,1.277,1.398
Best Maximum Sharpe,1.484,1.519,1.540,1.552
Best Minimum ES,1.135,1.185,1.279,1.424
Best Risk Parity,1.422,1.423,1.436,1.448
Equal Weight Benchmark,4.447,4.644,4.567,4.524


In [62]:
(
    evt_shape_sensitivity
    .T
    .plot(
        figsize=(10, 6),
        marker="o"
    )
)

plt.xlabel("Threshold Quantile")
plt.ylabel("GPD Shape Parameter")
plt.title("EVT Shape Parameter Stability Across Thresholds")
plt.grid(True, alpha=0.3)
plt.legend(
    title="Portfolio",
    bbox_to_anchor=(1.02, 1),
    loc="upper left"
)
plt.tight_layout()

plt.savefig(
    FIGURE_DIR / "evt_shape_threshold_sensitivity.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

C:\Users\Jani\AppData\Local\Temp\ipykernel_15712\1428709026.py:27: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [63]:
(
    evt_es_sensitivity
    * 100
).T.plot(
    figsize=(10, 6),
    marker="o"
)

plt.xlabel("Threshold Quantile")
plt.ylabel("EVT ES 99% (%)")
plt.title("99% EVT Expected Shortfall Sensitivity to Threshold Choice")
plt.grid(True, alpha=0.3)
plt.legend(
    title="Portfolio",
    bbox_to_anchor=(1.02, 1),
    loc="upper left"
)
plt.tight_layout()

plt.savefig(
    FIGURE_DIR / "evt_es99_threshold_sensitivity.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

C:\Users\Jani\AppData\Local\Temp\ipykernel_15712\4262100471.py:26: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [64]:
shortlist_export = shortlist.copy()

shortlist_export[
    "Weights Array"
] = shortlist_export[
    "Weights Array"
].apply(
    lambda x: ";".join(
        f"{w:.10f}"
        for w in x
    )
)

shortlist_export.to_csv(
    TABLE_DIR / "risk_shortlist_development.csv",
    index=False
)

print("Saved shortlist.")

Saved shortlist.


In [65]:
historical_risk.to_csv(
    TABLE_DIR / "historical_var_es_development.csv",
    index=False
)

historical_summary.to_csv(
    TABLE_DIR / "historical_var_es_summary.csv"
)

print("Saved Historical VaR and ES outputs.")

Saved Historical VaR and ES outputs.


In [66]:
parametric_risk.to_csv(
    TABLE_DIR / "parametric_var_es_development.csv",
    index=False
)

parametric_summary.to_csv(
    TABLE_DIR / "parametric_var_es_summary.csv"
)

print("Saved Parametric VaR and ES outputs.")

Saved Parametric VaR and ES outputs.


In [67]:
monte_carlo_risk.to_csv(
    TABLE_DIR / "monte_carlo_var_es_development.csv",
    index=False
)

monte_carlo_summary.to_csv(
    TABLE_DIR / "monte_carlo_var_es_summary.csv"
)

print("Saved Monte Carlo VaR and ES outputs.")

Saved Monte Carlo VaR and ES outputs.


In [68]:
var_model_comparison.to_csv(
    TABLE_DIR / "var_model_comparison.csv"
)

var_model_gap.to_csv(
    TABLE_DIR / "var_99_model_gap.csv"
)

evt_model_comparison_99.to_csv(
    TABLE_DIR / "var_es_model_comparison_99.csv"
)

print("Saved model-comparison outputs.")

Saved model-comparison outputs.


In [69]:
evt_risk.to_csv(
    TABLE_DIR / "evt_risk_development.csv",
    index=False
)

evt_parameters.to_csv(
    TABLE_DIR / "evt_gpd_parameters.csv"
)

evt_var_table.to_csv(
    TABLE_DIR / "evt_var_summary.csv"
)

evt_es_table.to_csv(
    TABLE_DIR / "evt_es_summary.csv"
)

evt_sensitivity.to_csv(
    TABLE_DIR / "evt_threshold_sensitivity.csv",
    index=False
)

evt_shape_sensitivity.to_csv(
    TABLE_DIR / "evt_shape_sensitivity.csv"
)

evt_var_sensitivity.to_csv(
    TABLE_DIR / "evt_var99_threshold_sensitivity.csv"
)

evt_es_sensitivity.to_csv(
    TABLE_DIR / "evt_es99_threshold_sensitivity.csv"
)

print("Saved EVT outputs.")

Saved EVT outputs.


In [70]:
final_risk_summary = pd.DataFrame({

    "Historical VaR 99%":
        historical_var_table[
            "VaR 99%"
        ],

    "Historical ES 99%":
        historical_es_table[
            "ES 99%"
        ],

    "Parametric VaR 99%":
        parametric_var_table[
            "Parametric VaR 99%"
        ],

    "Parametric ES 99%":
        parametric_es_table[
            "Parametric ES 99%"
        ],

    "Monte Carlo VaR 99%":
        monte_carlo_var_table[
            "Monte Carlo VaR 99%"
        ],

    "Monte Carlo ES 99%":
        monte_carlo_es_table[
            "Monte Carlo ES 99%"
        ],

    "EVT VaR 99%":
        evt_var_table[
            "EVT VaR 99%"
        ],

    "EVT ES 99%":
        evt_es_table[
            "EVT ES 99%"
        ]
})

(
    final_risk_summary
    * 100
).round(3)

,Historical VaR 99%,Historical ES 99%,Parametric VaR 99%,Parametric ES 99%,Monte Carlo VaR 99%,Monte Carlo ES 99%,EVT VaR 99%,EVT ES 99%
Portfolio,,,,,,,,
Best GMVP,0.649,1.269,0.661,0.760,0.654,0.752,0.668,1.277
Best Maximum Sharpe,0.971,1.540,0.878,1.010,0.877,1.015,0.983,1.540
Best Minimum ES,0.647,1.267,0.661,0.760,0.662,0.760,0.667,1.279
Best Risk Parity,0.939,1.458,0.848,0.976,0.849,0.976,0.958,1.436
Equal Weight Benchmark,3.029,4.469,2.376,2.728,2.372,2.723,2.972,4.567


In [71]:
final_risk_summary.to_csv(
    TABLE_DIR / "final_tail_risk_summary.csv"
)

print("Saved final tail-risk summary.")

Saved final tail-risk summary.


In [72]:
evt_threshold_diagnostics = (
    evt_sensitivity[
        [
            "Portfolio",
            "Threshold Quantile",
            "Threshold",
            "Exceedances",
            "Shape",
            "Scale",
            "EVT VaR 99%",
            "EVT ES 99%"
        ]
    ]
    .copy()
)

evt_threshold_diagnostics[
    "Exceedance Share"
] = (
    evt_threshold_diagnostics[
        "Exceedances"
    ]
    / len(returns_train)
)

evt_threshold_diagnostics

,Portfolio,Threshold Quantile,Threshold,Exceedances,Shape,Scale,EVT VaR 99%,EVT ES 99%,Exceedance Share
0,Equal Weight Benchmark,0.900,0.009836,431,0.223649,0.006701,0.030021,0.044468,0.100023
1,Equal Weight Benchmark,0.925,0.012084,324,0.312405,0.006198,0.029506,0.046435,0.075191
2,Equal Weight Benchmark,0.950,0.014697,216,0.275992,0.007400,0.029722,0.045670,0.050128
3,Equal Weight Benchmark,0.975,0.020521,108,0.253940,0.009064,0.029902,0.045244,0.025064
4,Best GMVP,0.900,0.002343,431,0.325045,0.001376,0.007057,0.011366,0.100023
5,Best GMVP,0.925,0.002856,324,0.419786,0.001258,0.006848,0.011903,0.075191
6,Best GMVP,0.950,0.003446,216,0.514287,0.001290,0.006684,0.012769,0.050128
7,Best GMVP,0.975,0.004603,108,0.634799,0.001523,0.006503,0.013976,0.025064
8,Best Maximum Sharpe,0.900,0.003628,431,0.211863,0.002226,0.010235,0.014835,0.100023
9,Best Maximum Sharpe,0.925,0.004422,324,0.299660,0.002000,0.009965,0.015193,0.075191


In [73]:
evt_threshold_display = (
    evt_threshold_diagnostics
    .copy()
)

evt_threshold_display[
    "Threshold"
] *= 100

evt_threshold_display[
    "Exceedance Share"
] *= 100

evt_threshold_display[
    "EVT VaR 99%"
] *= 100

evt_threshold_display[
    "EVT ES 99%"
] *= 100

evt_threshold_display.round(3)

,Portfolio,Threshold Quantile,Threshold,Exceedances,Shape,Scale,EVT VaR 99%,EVT ES 99%,Exceedance Share
0,Equal Weight Benchmark,0.900,0.984,431,0.224,0.007,3.002,4.447,10.002
1,Equal Weight Benchmark,0.925,1.208,324,0.312,0.006,2.951,4.644,7.519
2,Equal Weight Benchmark,0.950,1.470,216,0.276,0.007,2.972,4.567,5.013
3,Equal Weight Benchmark,0.975,2.052,108,0.254,0.009,2.990,4.524,2.506
4,Best GMVP,0.900,0.234,431,0.325,0.001,0.706,1.137,10.002
5,Best GMVP,0.925,0.286,324,0.420,0.001,0.685,1.190,7.519
6,Best GMVP,0.950,0.345,216,0.514,0.001,0.668,1.277,5.013
7,Best GMVP,0.975,0.460,108,0.635,0.002,0.650,1.398,2.506
8,Best Maximum Sharpe,0.900,0.363,431,0.212,0.002,1.024,1.484,10.002
9,Best Maximum Sharpe,0.925,0.442,324,0.300,0.002,0.997,1.519,7.519


In [74]:
evt_threshold_stability = (
    evt_sensitivity
    .groupby(
        "Threshold Quantile"
    )
    .agg(
        Mean_Exceedances=(
            "Exceedances",
            "mean"
        ),
        Mean_Shape=(
            "Shape",
            "mean"
        ),
        Shape_Std=(
            "Shape",
            "std"
        ),
        Mean_EVT_VaR_99=(
            "EVT VaR 99%",
            "mean"
        ),
        Mean_EVT_ES_99=(
            "EVT ES 99%",
            "mean"
        )
    )
)

evt_threshold_stability

,Mean_Exceedances,Mean_Shape,Shape_Std,Mean_EVT_VaR_99,Mean_EVT_ES_99
Threshold Quantile,,,,,
0.900,431.0,0.261518,0.059058,0.012832,0.019248
0.925,324.0,0.335211,0.082171,0.012588,0.019922
0.950,216.0,0.389108,0.119254,0.012497,0.020198
0.975,108.0,0.458608,0.177764,0.012407,0.020691


## EVT Threshold Selection Justification

The Peaks Over Threshold approach requires a balance between tail relevance
and estimation reliability.

A lower threshold provides more exceedances and therefore more observations
for estimation, but it may include losses that are not sufficiently extreme
for the Generalized Pareto approximation.

A higher threshold isolates more extreme losses but produces fewer
exceedances and therefore greater parameter uncertainty.

Thresholds at the 90th, 92.5th, 95th and 97.5th loss percentiles were tested.

The fitted GPD shape parameter is not perfectly stable across thresholds for
all portfolios. Therefore, threshold selection is not based on parameter
stability alone.

The 95th-percentile threshold is retained as a practical compromise. It
focuses the model on the extreme-loss region while retaining approximately
216 exceedances per portfolio, compared with only about 108 observations at
the 97.5th-percentile threshold.

The resulting 99% EVT VaR estimates are relatively stable across nearby
thresholds, while Expected Shortfall is more sensitive for some defensive
portfolios. This sensitivity is explicitly retained as part of the model
risk assessment.

Therefore, the 95th-percentile threshold is selected as the baseline EVT
specification, while the surrounding thresholds are retained as robustness
checks rather than being ignored.

In [75]:
evt_threshold_diagnostics.to_csv(
    TABLE_DIR
    / "evt_threshold_diagnostics.csv",
    index=False
)

evt_threshold_stability.to_csv(
    TABLE_DIR
    / "evt_threshold_stability_summary.csv"
)

print(
    "Saved EVT threshold diagnostic outputs."
)

Saved EVT threshold diagnostic outputs.


In [76]:
var_model_comparison_95.to_csv(
    TABLE_DIR
    / "var_model_comparison_95.csv"
)

es_model_comparison_95.to_csv(
    TABLE_DIR
    / "es_model_comparison_95.csv"
)

In [77]:
assert len(shortlist) == 5

assert len(historical_risk) == 15
assert len(parametric_risk) == 15
assert len(monte_carlo_risk) == 15

assert len(evt_risk) == 10
assert len(evt_sensitivity) == 20

assert (
    historical_risk[
        "ES >= VaR"
    ].all()
)

assert (
    parametric_risk[
        "ES >= VaR"
    ].all()
)

assert (
    monte_carlo_risk[
        "ES >= VaR"
    ].all()
)

assert (
    evt_risk[
        "ES >= VaR"
    ].all()
)

assert (
    evt_risk[
        "Shape"
    ]
    < 1
).all()

assert (
    returns_train.shape
    == (
        4309,
        7
    )
)

assert (
    returns_train.index.max()
    == pd.Timestamp(
        "2021-12-31"
    )
)

assert (
    shortlist[
        "Weights Array"
    ]
    .apply(
        lambda w:
        np.isclose(
            w.sum(),
            1.0,
            atol=1e-6
        )
    )
    .all()
)

print(
    "All Notebook 05 final "
    "validation checks passed."
)

All Notebook 05 final validation checks passed.
